In [1]:
import pandas as pd
import geopandas as gpd
from shapely.geometry import Point
from tqdm import tqdm
#import contextily as ctxv
import osmnx as ox
from osmnx import graph_to_gdfs
from shapely.ops import unary_union
from shapely.geometry import MultiPoint
import logging 
from geopandas import sjoin
import pickle
ox.settings.log_console = True

In [2]:
from shapely.geometry import Polygon, MultiPoint
import math
import libpysal
# from sklearn.cluster import AgglomerativeClustering # Not used if using adaptive regionalization
from tqdm.auto import tqdm
import logging
import pickle
import matplotlib.pyplot as plt
from libpysal.weights import KNN, DistanceBand, Queen # Queen for low-density merging
from matplotlib.colors import LogNorm, Normalize
import numpy as np # For np.nan checking if needed, though fillna is better

# --- Initial Setup ---
logging.basicConfig(level=logging.INFO, # Changed to INFO for less verbose default, DEBUG for dev
                    format='%(asctime)s - %(levelname)s - %(filename)s:%(lineno)d - %(message)s')

ox.settings.log_console = True
logging.info("Script started.")

2025-06-13 20:57:43,312 - INFO - 2105997383.py:18 - Script started.


In [3]:
logging.info("Step 1: Loading data and creating initial GeoDataFrame...")
df = pd.read_csv("hurricane_matrix.csv")
df.head()
df = df.rename(columns={"Unnamed: 0": "traj_id"})
# 2.2. Set traj_id as the DataFrame index
df = df.set_index("traj_id")
df_long = df.stack().rename("coords").reset_index()

df_long = df_long.rename(columns={"level_1": "pt_idx"})
print(df_long)

2025-06-13 20:57:43,315 - INFO - 1812683755.py:1 - Step 1: Loading data and creating initial GeoDataFrame...


                                                   traj_id pt_idx  \
0        e464d76fd1b9ea6e0c135c0292581eccc3f27306fb76af...      1   
1        e464d76fd1b9ea6e0c135c0292581eccc3f27306fb76af...      3   
2        e464d76fd1b9ea6e0c135c0292581eccc3f27306fb76af...      5   
3        e464d76fd1b9ea6e0c135c0292581eccc3f27306fb76af...     20   
4        e464d76fd1b9ea6e0c135c0292581eccc3f27306fb76af...     21   
...                                                    ...    ...   
1677305  559f9e9acaaf7e32ea1c7b2616af890a63a5fd8ebd2a54...    130   
1677306  559f9e9acaaf7e32ea1c7b2616af890a63a5fd8ebd2a54...    133   
1677307  559f9e9acaaf7e32ea1c7b2616af890a63a5fd8ebd2a54...    134   
1677308  559f9e9acaaf7e32ea1c7b2616af890a63a5fd8ebd2a54...    140   
1677309  559f9e9acaaf7e32ea1c7b2616af890a63a5fd8ebd2a54...    141   

                          coords  
0        (24.548312, -81.796707)  
1        (24.548324, -81.796739)  
2        (24.548307, -81.796648)  
3        (26.451009, -81.949365

In [4]:
import ast

# (A) Define a helper that tries to literal_eval the string,
#     and falls back to (nan, nan) on any failure.
def parse_coord_string(s):
    """
    Convert a string '(lat, lon)' into a (float(lat), float(lon)) tuple.
    If parsing fails, return (nan, nan).
    """
    if not isinstance(s, str):
        # If somehow it's already a tuple/list, cast its elements to float
        if isinstance(s, (tuple, list)) and len(s) == 2:
            try:
                return (float(s[0]), float(s[1]))
            except:
                return (np.nan, np.nan)
        return (np.nan, np.nan)

    try:
        # literal_eval safely parses Python literal syntax
        parsed = ast.literal_eval(s)
        if isinstance(parsed, (tuple, list)) and len(parsed) == 2:
            lat, lon = float(parsed[0]), float(parsed[1])
            return (lat, lon)
        else:
            return (np.nan, np.nan)
    except Exception:
        return (np.nan, np.nan)
print(df_long)
# (B) Apply this to your entire df_long["coords"] column
df_long["coords"] = df_long["coords"].apply(parse_coord_string)

# (C) Verify all entries are now tuples of two floats
print("After parsing, types in df_long['coords']:")
print(df_long["coords"].apply(type).value_counts())

                                                   traj_id pt_idx  \
0        e464d76fd1b9ea6e0c135c0292581eccc3f27306fb76af...      1   
1        e464d76fd1b9ea6e0c135c0292581eccc3f27306fb76af...      3   
2        e464d76fd1b9ea6e0c135c0292581eccc3f27306fb76af...      5   
3        e464d76fd1b9ea6e0c135c0292581eccc3f27306fb76af...     20   
4        e464d76fd1b9ea6e0c135c0292581eccc3f27306fb76af...     21   
...                                                    ...    ...   
1677305  559f9e9acaaf7e32ea1c7b2616af890a63a5fd8ebd2a54...    130   
1677306  559f9e9acaaf7e32ea1c7b2616af890a63a5fd8ebd2a54...    133   
1677307  559f9e9acaaf7e32ea1c7b2616af890a63a5fd8ebd2a54...    134   
1677308  559f9e9acaaf7e32ea1c7b2616af890a63a5fd8ebd2a54...    140   
1677309  559f9e9acaaf7e32ea1c7b2616af890a63a5fd8ebd2a54...    141   

                          coords  
0        (24.548312, -81.796707)  
1        (24.548324, -81.796739)  
2        (24.548307, -81.796648)  
3        (26.451009, -81.949365

In [5]:
bad_coords = df_long["coords"].apply(lambda x: not (isinstance(x, (tuple, list)) and len(x) == 2))
if bad_coords.any():
    # If any row is not a 2‐tuple, inspect them:
    print("Rows with malformed coords:")
    print(df_long.loc[bad_coords, :].head())
    raise ValueError("Some coords are not length‐2 tuples. Please inspect your data.")
else:
    logging.info("All coords appear to be 2‐element tuples. Proceed to unpack.")

# 4.2. Unpack into two new columns, latitude & longitude
df_long[["latitude", "longitude"]] = pd.DataFrame(
    df_long["coords"].tolist(), index=df_long.index, columns=["latitude", "longitude"]
)
df_long = df_long.drop(columns="coords")

# 4.3. Convert pt_idx (if it’s still a string) into an integer from 1..144
#      If your original wide columns were strings '1','2',… you can safely convert:
try:
    df_long["pt_idx"] = df_long["pt_idx"].astype(int)
except ValueError:
    # If pt_idx cannot cast to int (e.g. if columns were named "STEP1"), leave as-is
    pass

print("After unpacking, columns of df_long:", df_long.columns.tolist())
print(df_long)

2025-06-13 20:57:58,481 - INFO - 2322511903.py:8 - All coords appear to be 2‐element tuples. Proceed to unpack.


After unpacking, columns of df_long: ['traj_id', 'pt_idx', 'latitude', 'longitude']
                                                   traj_id  pt_idx   latitude  \
0        e464d76fd1b9ea6e0c135c0292581eccc3f27306fb76af...       1  24.548312   
1        e464d76fd1b9ea6e0c135c0292581eccc3f27306fb76af...       3  24.548324   
2        e464d76fd1b9ea6e0c135c0292581eccc3f27306fb76af...       5  24.548307   
3        e464d76fd1b9ea6e0c135c0292581eccc3f27306fb76af...      20  26.451009   
4        e464d76fd1b9ea6e0c135c0292581eccc3f27306fb76af...      21  26.451007   
...                                                    ...     ...        ...   
1677305  559f9e9acaaf7e32ea1c7b2616af890a63a5fd8ebd2a54...     130  29.942492   
1677306  559f9e9acaaf7e32ea1c7b2616af890a63a5fd8ebd2a54...     133  30.080000   
1677307  559f9e9acaaf7e32ea1c7b2616af890a63a5fd8ebd2a54...     134  30.080000   
1677308  559f9e9acaaf7e32ea1c7b2616af890a63a5fd8ebd2a54...     140  29.944345   
1677309  559f9e9acaaf7e32

In [6]:
df_long["geometry"] = df_long.apply(
    lambda r: Point(r.longitude, r.latitude),
    axis=1
)
point_gdf = gpd.GeoDataFrame(
    df_long[["traj_id", "pt_idx", "latitude", "longitude", "geometry"]],
    geometry="geometry",
    crs="EPSG:4326"
)
logging.info(f"Built point_gdf with {len(point_gdf)} points.")


2025-06-13 20:58:21,808 - INFO - 554215537.py:10 - Built point_gdf with 1677310 points.


In [7]:
## We can load from Hex_cent.
# ————————————————————————————————————————————————
list_points = []
for xy in tqdm(zip(df_long.longitude, df_long.latitude), desc="Preparing points for hull", total=len(df)):
    list_points.append((xy[0],xy[1]))

if not list_points or len(list_points) < 3:
    logging.error("Not enough points (<3) for hull. Exiting.")
    exit()

all_points_multipoint = MultiPoint(list_points)
points_crs = point_gdf.crs
s = gpd.GeoSeries([all_points_multipoint], crs=points_crs)

concavity_ratio = 0.05
allow_holes_param = False
logging.info(f"Creating GeoPandas concave hull with ratio={concavity_ratio} and allow_holes={allow_holes_param}...")
study_poly = None
study_poly_geoseries = s.concave_hull(ratio=concavity_ratio, allow_holes=allow_holes_param)
if not study_poly_geoseries.empty and study_poly_geoseries.iloc[0] is not None and not study_poly_geoseries.iloc[0].is_empty:
    study_poly = study_poly_geoseries.iloc[0] # returns the polygon of concave hull

Preparing points for hull:   0%|          | 0/95410 [00:00<?, ?it/s]

2025-06-13 20:58:39,306 - INFO - 4063424718.py:17 - Creating GeoPandas concave hull with ratio=0.05 and allow_holes=False...


In [8]:
hex_gdf = pd.read_parquet("")
hex_gdf = hex_gdf.reset_index(drop=True)  

# 2) Create a hex_id column (string or integer) from that index
#    You can keep it as an integer, or convert to string, e.g. "H0", "H1", ...
hex_gdf["hex_id"] = hex_gdf.index.astype(str)

# Let’s say hex_df["geometry"] is dtype=object of WKB bytes.
hex_gdf["geometry"] = gpd.GeoSeries.from_wkb(hex_gdf["geometry"])
# Now convert to a GeoDataFrame:
hex_gdf = gpd.GeoDataFrame(hex_gdf, geometry="geometry", crs="EPSG:4326")

print(hex_gdf.head())



                                            geometry hex_id
0  POLYGON ((-81.8 24.45908, -81.79741 24.4546, -...      0
1  POLYGON ((-81.795 24.45908, -81.79236 24.45908...      1
2  POLYGON ((-81.8175 24.4764, -81.8075 24.4764, ...      2
3  POLYGON ((-81.7925 24.45908, -81.8025 24.45908...      3
4  POLYGON ((-81.7925 24.46774, -81.7875 24.4764,...      4


In [9]:
FL_POI_DS = pd.read_csv("Hex_bound_POI.csv")
FL_POI_DS["geometry"] = gpd.points_from_xy(
    FL_POI_DS["LONGITUDE"].astype(float),
    FL_POI_DS["LATITUDE"].astype(float)
)

# 2. If FL_POI_DS was not already a GeoDataFrame, turn it into one:
FL_POI_GDF = gpd.GeoDataFrame(
    FL_POI_DS,
    geometry="geometry",
    crs="EPSG:4326"   # (or whatever CRS these lon/lat coordinates use)
)
FL_POI_GDF.head

import matplotlib.pyplot as plt
import geopandas as gpd

# Assuming hex_gdf and study_poly are already defined in the environment.
# If study_poly is a Shapely polygon, convert to GeoDataFrame:
hull_gdf = gpd.GeoDataFrame({'geometry': [study_poly]}, crs=hex_gdf.crs)



In [10]:
FL_POI_GDF['concatenated'] = (
    FL_POI_GDF['concatenated']
      .str.split(r'\[sep\]')    # split on literal “[sep]”
      .str[:2]                  # keep only the first two pieces
      .str.join('[sep]')        # re-join with “[sep]”
)
mask = FL_POI_GDF.within(study_poly)
Hull_FL_POI_GDF=FL_POI_GDF[mask].reset_index()
print(Hull_FL_POI_GDF)

           index  Unnamed: 0             PLACEKEY  LONGITUDE   LATITUDE  \
0              0           4  zzy-22s@8dj-jv6-ks5 -82.524818  27.949076   
1              1           6  zzy-22s@8gp-3nn-3dv -81.380769  30.199156   
2              2          12  zzy-22t@8dh-h4q-gx5 -81.769293  24.566605   
3              3          27  zzy-234@8gp-2dc-fj9 -81.682551  30.203123   
4              4         157  zzz-222@8dg-kpy-btv -81.847107  26.642022   
...          ...         ...                  ...        ...        ...   
1548012  1642499    20309583  zzy-222@8gq-r2j-t5f -81.426410  29.784013   
1548013  1642500    20309584  zzy-222@8gq-tkh-qcq -81.070241  29.217953   
1548014  1642501    20309585  zzy-222@8gq-tsg-5vf -81.322792  29.042548   
1548015  1642502    20309586  zzy-222@8gq-wfw-yd9 -81.019375  29.249829   
1548016  1642503    20309587  zzy-222@8gq-wwk-fj9 -80.988885  29.145973   

                                              concatenated REGION  \
0        Activities Related to

In [11]:
hex_gdf = hex_gdf.set_crs(epsg=4326, allow_override=True)

# 2) reproject to an equal‐area CRS
hex_aea = hex_gdf.to_crs(epsg=5070)

# 3) compute true planar centroids
hex_aea['centroid_proj'] = hex_aea.geometry.centroid

# 4) extract X/Y in projected units if you need them:
hex_aea['centroid_x_proj'] = hex_aea.centroid_proj.x
hex_aea['centroid_y_proj'] = hex_aea.centroid_proj.y

# 5) if you still want lon/lat centroids, reproject back
centroids_ll = hex_aea.centroid_proj.to_crs(epsg=4326)
hex_gdf['centroid']   = centroids_ll
hex_gdf['centroid_x'] = centroids_ll.x
hex_gdf['centroid_y'] = centroids_ll.y

In [12]:
Hull_FL_POI_GDF = gpd.GeoDataFrame(Hull_FL_POI_GDF,geometry="geometry", crs="EPSG:4326")
Hull_proj = Hull_FL_POI_GDF.to_crs(epsg=5070)
hex_proj  = hex_gdf.to_crs(epsg=5070)

# 2) Do the nearest‐neighbor join in projected space
joined_proj = gpd.sjoin_nearest(
    Hull_proj[['concatenated','geometry']],
    hex_proj[['hex_id','geometry','centroid']],
    how='right',
    distance_col='dist_m'    # optional: records the actual distance in meters
)

# 3) If you need the joined table back in lon/lat:
joined_ll = joined_proj.to_crs(epsg=4326)

# 4) Clean up (drop the auto‐indices, etc.)
joined = (
    joined_ll
      .drop(columns=['index_left'])
      #.set_index('hex_id')
)


In [20]:
import torch
from transformers import AutoTokenizer, AutoModelForCausalLM
from peft import PeftModel
# 4) Move to GPU/CPU and set eval mode:
device = torch.device("cuda" if torch.cuda.is_available() else "cpu")
# 1) Where you saved your LoRA adapters after training:
CHECKPOINT = "/storage1/fs1/nlin/Active/sizhe/FO_DATA/checkpoint-dir/checkpoint-551"

# 2) Load the tokenizer from that folder (it contains tokenizer.json + vocab.txt)
tokenizer = AutoTokenizer.from_pretrained(CHECKPOINT)

base_name = "google/gemma-3-1b-it"     # <— exactly what you passed in your training script
base = AutoModelForCausalLM.from_pretrained(
    "google/gemma-3-1b-it",
    attn_implementation="eager"
).to(device)

# 3) Now graft the adapter weights:
model = PeftModel.from_pretrained(base, CHECKPOINT)


model.to(device).eval()



/storage1/fs1/nlin/Active/sizhe/sizhe/envs/geospatial/lib/python3.12/site-packages/peft/peft_model.py:565: UserWarning: Found missing adapter keys while loading the checkpoint: ['base_model.model.model.layers.0.self_attn.q_proj.lora_A.default.weight', 'base_model.model.model.layers.0.self_attn.q_proj.lora_B.default.weight', 'base_model.model.model.layers.0.self_attn.k_proj.lora_A.default.weight', 'base_model.model.model.layers.0.self_attn.k_proj.lora_B.default.weight', 'base_model.model.model.layers.0.self_attn.v_proj.lora_A.default.weight', 'base_model.model.model.layers.0.self_attn.v_proj.lora_B.default.weight', 'base_model.model.model.layers.0.self_attn.o_proj.lora_A.default.weight', 'base_model.model.model.layers.0.self_attn.o_proj.lora_B.default.weight', 'base_model.model.model.layers.1.self_attn.q_proj.lora_A.default.weight', 'base_model.model.model.layers.1.self_attn.q_proj.lora_B.default.weight', 'base_model.model.model.layers.1.self_attn.k_proj.lora_A.default.weight', 'base_mo

PeftModelForCausalLM(
  (base_model): LoraModel(
    (model): Gemma3ForCausalLM(
      (model): Gemma3TextModel(
        (embed_tokens): Gemma3TextScaledWordEmbedding(262144, 1152, padding_idx=0)
        (layers): ModuleList(
          (0-25): 26 x Gemma3DecoderLayer(
            (self_attn): Gemma3Attention(
              (q_proj): lora.Linear(
                (base_layer): Linear(in_features=1152, out_features=1024, bias=False)
                (lora_dropout): ModuleDict(
                  (default): Dropout(p=0.1, inplace=False)
                )
                (lora_A): ModuleDict(
                  (default): Linear(in_features=1152, out_features=16, bias=False)
                )
                (lora_B): ModuleDict(
                  (default): Linear(in_features=16, out_features=1024, bias=False)
                )
                (lora_embedding_A): ParameterDict()
                (lora_embedding_B): ParameterDict()
                (lora_magnitude_vector): ModuleDict()
         

In [ ]:
def embed_texts(texts: list[str]) -> torch.Tensor:
    """
    texts: a Python list of strings (each may contain literal "[sep]").
    example: ["[Museums, Historical Sites,...<sep>Museums, Historical Sites<sep>Natural History museum]",...]
    In the format of [<category><sep><subcategory><sep><name>]
    Returns: a GPU tensor of shape (len(texts), hidden_size) containing the
             final‐token embedding for each string.
    """
    # 2.1) Clean & force everything to str, replacing None/NaN with ""
    clean_texts=[]
    for t in texts:
        if t is None:
            clean_texts.append("")
        else:
            clean_texts.append(str(t))
    sep = tokenizer.sep_token  # e.g. "[SEP]" for BERT‐style; whatever your tokenizer.sep_token is
    clean_texts = [t.replace("[sep]", sep) for t in clean_texts]
    
    # 2.3) Batch‐tokenize all strings onto GPU
    enc = tokenizer(
        clean_texts,
        return_tensors="pt",
        padding=True,
        truncation=True,
        max_length=512,
        return_token_type_ids=False,
    ).to(device)
    #print("Batch input_ids are on:", enc.input_ids.device)
    # Now enc.input_ids and enc.attention_mask live on GPU.

    # 2.4) Forward‐pass (no gradients) to get hidden states
    with torch.no_grad():
        outputs = model.base_model(
            input_ids=enc.input_ids,
            attention_mask=enc.attention_mask,
            output_hidden_states=True
        )
        # `outputs.hidden_states` is a tuple of length (num_layers+1);
        # each element has shape (batch_size, seq_len, hidden_size).
        last_hidden = outputs.hidden_states[-1]  # (batch_size, seq_len, hidden_size) on GPU

    # 2.5) For each sequence, pick out the final non‐pad token embedding
    seq_lens = enc.attention_mask.sum(dim=1) - 1  # (batch_size,) on GPU, index of last non-pad
    batch_size, hidden_size = last_hidden.size(0), last_hidden.size(2)

    # Gather the embedding at position (i, seq_lens[i], :)
    final_embs = last_hidden[torch.arange(batch_size), seq_lens, :]  # (batch_size, hidden_size) on GPU
    #print((final_embs))
    return final_embs


In [ ]:
FL_POI_GDF['concatenated_short'] = (
    FL_POI_GDF['concatenated']
      .str.split(r'\[sep\]')    # split on literal “[sep]”
      .str[:2]                  # keep only the first two pieces
      .str.join('[sep]')        # re-join with “[sep]”
)
unique_labels = FL_POI_GDF['concatenated_short'].unique().tolist()
rows = []
for text in unique_labels:
    row ={
        "label" : text,
        "embedding" : embed_texts([text]).cpu().numpy()[0]
    }
    rows.append(row)
df_labels = pd.DataFrame(rows)

    
    

In [1]:
import pyarrow.parquet as pq
import pandas as pd

def read_parquet_in_rowgroups(path, columns=None):
    pf = pq.ParquetFile(path)
    parts = []
    for rg in range(pf.num_row_groups):
        tbl = pf.read_row_group(rg, columns=columns)
        parts.append(tbl.to_pandas())
    return pd.concat(parts, ignore_index=True)

# Only pull in the vector columns (drop geometry if you don’t need it)
cols = [c for c in pq.ParquetFile("POI_vec_proj_matrix.parquet").schema.names
        if c not in ("geometry",)]
vec_proj_matrix = read_parquet_in_rowgroups(
    "POI_vec_proj_matrix.parquet",
    columns=cols
)


In [2]:
vec_proj_matrix

,concatenated,REGION,LOCATION_NAME,concatenated_vec
0,Activities Related to Real Estate[sep]Resident...,FL,Jeffrey Berg,[-3.41533518e+00 -1.49931777e+00 1.36452806e+...
1,Offices of Real Estate Agents and Brokers[sep]...,FL,Search Beacon Realty,[-2.38089347e+00 2.18140173e+00 -8.13243628e-...
2,"Sporting Goods, Hobby, and Musical Instrument ...",FL,Bone Island Music Key West,[ 5.42108893e-01 1.63252306e+00 -1.99507982e-...
3,Offices of Dentists[sep]Offices of Dentists[se...,FL,Dr. Arbel Maghsoodpour DDS,[-3.0437524e+00 -1.1710362e+00 6.8726033e-01 ...
4,"Museums, Historical Sites, and Similar Institu...",FL,Roberto Clemente Park,[ 2.64899683e+00 4.96178687e-01 -9.97523606e-...
...,...,...,...,...
1548012,Home Furnishings Stores[sep]All Other Home Fur...,FL,FL Custom Kitchen and Baths,[ 5.19509935e+00 -3.56546664e+00 1.36108625e+...
1548013,Urban Transit Systems[sep]Bus and Other Motor ...,FL,Votran Cedar Highlands & Jimmy Ann Ib,[ 2.06519341e+00 -2.15121603e+00 1.27448708e-...
1548014,Urban Transit Systems[sep]Bus and Other Motor ...,FL,Votran Plymouth & 15a Ib,[ 2.22038412e+00 -8.90327156e-01 4.48130012e-...
1548015,Urban Transit Systems[sep]Bus and Other Motor ...,FL,Votran Zelda & A1a Ib,[ 7.69490540e-01 -3.18823189e-01 3.16257668e+...


In [ ]:
vec_proj_matrix
#print(len(ast.literal_eval(vec_proj_matrix["concatenated_vec"][0])))
s = vec_proj_matrix["concatenated_vec"][0]

# 1) strip off the brackets
s_clean = s.strip("[]")

# 2) parse into a 1D numpy array
arr = np.fromstring(s_clean, sep=" ")
print(len(arr))

In [ ]:
import numpy as np
import pandas as pd
from sklearn.preprocessing import LabelEncoder
from torch.utils.data import Dataset, DataLoader
import torch
import torch.nn as nn
import torch.nn.functional as F

# ── 1) HYPERPARAMETERS ─────────────────────────────────────────────────────────
LATENT_DIM   = 6     # target lower dimension
HIDDEN_DIM   = 256     # hidden size of MLP
BATCH_SIZE   = 128
LR           = 1e-3
EPOCHS       = 50
DEVICE       = torch.device("cuda" if torch.cuda.is_available() else "cpu")

# ── 2) PREPARE DATA ────────────────────────────────────────────────────────────
# Extract X matrix and integer labels y
X = np.stack(df_labels['embedding'].values)               # (N, D)
le = LabelEncoder()
y = le.fit_transform(df_labels['label'].values)           # (N,)
n_classes = len(le.classes_)

# PyTorch Dataset
class EmbedDataset(Dataset):
    def __init__(self, X, y):
        self.X = torch.from_numpy(X).float()
        self.y = torch.from_numpy(y).long()
    def __len__(self):
        return len(self.y)
    def __getitem__(self, i):
        return self.X[i], self.y[i]

ds    = EmbedDataset(X, y)
loader= DataLoader(ds, batch_size=BATCH_SIZE, shuffle=True, num_workers=0)

# ── 3) MODEL DEFINITION ────────────────────────────────────────────────────────
class BottleneckMLP(nn.Module):
    def __init__(self, in_dim, hid_dim, lat_dim, n_cls):
        super().__init__()
        self.encoder = nn.Sequential(
            nn.Linear(in_dim, hid_dim),
            nn.ReLU(inplace=True),
            nn.Linear(hid_dim, lat_dim),
            nn.ReLU(inplace=True)
        )
        self.head = nn.Linear(lat_dim, n_cls)
    def forward(self, x):
        z      = self.encoder(x)
        logits = self.head(z)
        return z, logits

model = BottleneckMLP(
    in_dim=X.shape[1],
    hid_dim=HIDDEN_DIM,
    lat_dim=LATENT_DIM,
    n_cls=n_classes
).to(DEVICE)

optimizer = torch.optim.Adam(model.parameters(), lr=LR)
criterion = nn.CrossEntropyLoss()

# ── 4) TRAIN ─────────────────────────────────────────────────────────────────
for epoch in range(1, EPOCHS+1):
    model.train()
    total_loss = 0
    for xb, yb in loader:
        xb, yb = xb.to(DEVICE), yb.to(DEVICE)
        optimizer.zero_grad()
        _, logits = model(xb)
        loss = criterion(logits, yb)
        loss.backward()
        optimizer.step()
        total_loss += loss.item() * xb.size(0)
    print(f"Epoch {epoch:2d}  loss = {total_loss/len(ds):.4f}")

# ── 5) EXTRACT LATENT CODES & APPEND TO df ────────────────────────────────────
model.eval()
with torch.no_grad():
    X_tensor = torch.from_numpy(X).float().to(DEVICE)
    Z_tensor, _ = model(X_tensor)               # (N, LATENT_DIM)
    Z = Z_tensor.cpu().numpy()

# Build a small DataFrame of the new features
cols_z   = [f"z_{i}" for i in range(LATENT_DIM)]
df_z      = pd.DataFrame(Z, columns=cols_z, index=df_labels.index)

# Concatenate back onto your original df
df_supervised = pd.concat([df_labels, df_z], axis=1)


In [ ]:
df_supervised

In [ ]:
# 1) Grab all the latent-dim columns
latent_cols = [c for c in df_supervised.columns if c.startswith("z_")]

# 2) Pack them into a new column “z” as a list per row
# Option A: as Python lists
df_supervised["z"] = df_supervised[latent_cols].values.tolist()

# Option B: as NumPy arrays
# df_supervised["z"] = list(df_supervised[latent_cols].to_numpy())

# 3) (Optional) drop the old columns
df_supervised.drop(columns=latent_cols, inplace=True)

# Now df_supervised has one column 'z' where each entry is your full 128-dim vector.
print(df_supervised.head())


In [ ]:
def parse_vec(s: str) -> np.ndarray:
    # strip leading/trailing brackets, then whitespace-split to floats
    return np.fromstring(s.strip("[]"), sep=" ")

vec_proj_matrix["vec_arr"] = vec_proj_matrix["concatenated_vec"].apply(parse_vec)

In [ ]:
import numpy as np
import torch

device = torch.device("cuda" if torch.cuda.is_available() else "cpu")
model.eval()

# Prepare your parsed POI arrays
X_poi = np.stack(vec_proj_matrix["vec_arr"].values)   # (M, D)

batch_size = 1000
Z_chunks = []
with torch.no_grad():
    # Optionally move model to CPU if you have no GPU
    # model.to("cpu"); device = torch.device("cpu")

    for start in tqdm(range(0, X_poi.shape[0], batch_size)):
        end = start + batch_size
        batch = X_poi[start:end]                        # shape (B, D)
        t     = torch.from_numpy(batch).float().to(device)
        z_b   = model.encoder(t)                        # (B, LATENT_DIM)
        Z_chunks.append(z_b.cpu().numpy())

# Concatenate all the chunks back together
Z_poi = np.vstack(Z_chunks)                          # (M, LATENT_DIM)
vec_proj_matrix["z_poi"] = list(Z_poi)


In [ ]:


# 4) Extract top-level category for each POI
vec_proj_matrix["category"] = vec_proj_matrix["concatenated"]\
                                  .str.split(r"\[sep\]").str[0]




# 6) Now run t-SNE **only on the top 10 categories** by count
top10 = vec_proj_matrix["category"]\
           .value_counts()\
           .nlargest(10)\
           .index.tolist()

sub = vec_proj_matrix[vec_proj_matrix["category"].isin(top10)].copy()
Z_sub = np.stack(sub["z_poi"].values)                # (N_sub, LATENT_DIM)

tsne = TSNE(
    n_components=2,
    perplexity=30,
    learning_rate="auto",
    init="random",
    random_state=42
)
Z2 = tsne.fit_transform(Z_sub)                       # (N_sub, 2)

# 7) Plot, colored by category
sub["cat_code"] = sub["category"].astype("category").cat.codes
cats = sub["category"].astype("category").cat.categories

plt.figure(figsize=(10, 8))
sc = plt.scatter(Z2[:, 0], Z2[:, 1], c=sub["cat_code"], alpha=0.7)
plt.title("t-SNE of Supervised Latents (Top 10 Categories)")
plt.xlabel("TSNE-1"); plt.ylabel("TSNE-2")

# build legend
handles = [
    plt.Line2D([], [], marker="o", linestyle="", 
               color=sc.cmap(sc.norm(i)), label=cat)
    for i, cat in enumerate(cats)
]
plt.legend(handles=handles, title="Category",
           bbox_to_anchor=(1.05, 1), loc="upper left",
           fontsize="small")
plt.tight_layout()
plt.show()
